# CU-BEMS Intelligence Suite — Question 1 (Operational Baseload)
## Notebook 1B: Raw Telemetry Ingestion (`qn1b_raw_ingestion.ipynb`)

---

### Scope Separation (Question 1 vs Question 2)
* **Question 1 Scope**: Strictly focused on **Lighting** (`_Light(kW)`) and **Plug Receptacles** (`_Plug(kW)`). These represent non-essential, schedule-dependent baseload.
* **Question 2 Scope (Separated)**: HVAC systems (`_Air(kW)`) and indoor thermal physics (temperature, humidity, CO2) are handled independently by the HVAC team.

### Pipeline Objectives
1. Connect to CU-BEMS raw repository via Kaggle API / kagglehub (with zero-download local fallback).
2. Filter exclusively for Lighting and Plug electrical telemetry circuits (excluding non-electrical Lux sensors and HVAC).
3. Concatenate the 2018 and 2019 data streams into continuous 18-month floor tables.
4. Persist clean intermediate raw extractions to `../data/raw_Floor{1-7}.csv`.

In [1]:
# ==============================================================================
# 1. ENVIRONMENT & SEARCH PATH SETUP (KAGGLE API + LOCAL FALLBACK)
# ==============================================================================
import os
import time
import pandas as pd
import numpy as np

notebook_dir = os.getcwd()
data_dir = os.path.abspath(os.path.join(notebook_dir, "..", "data"))

# Auto-pull directly from Kaggle Hub API cache if local files are absent
try:
    import kagglehub
    raw_source_dir = kagglehub.dataset_download("claytonmiller/cubems-smart-building-energy-and-iaq-data")
    print(f"[OK] Kaggle Hub API active. Data sourced from cache:\n  {raw_source_dir}")
except Exception as e:
    raw_source_dir = data_dir
    print(f"[INFO] Falling back to local data directory:\n  {raw_source_dir}")


[OK] Kaggle Hub API active. Data sourced from cache:
  C:\Users\65964\.cache\kagglehub\datasets\claytonmiller\cubems-smart-building-energy-and-iaq-data\versions\1


## 2. Ingesting Raw Telemetry (Floors 1 to 7: Lights and Plugs Only)
We process each floor, extracting solely the active electrical power circuits (`_Light(kW)` and `_Plug(kW)`) required for Question 1.

In [2]:
# ==============================================================================
# 2. INGESTION LOOP ACROSS FLOORS 1 TO 7
# ==============================================================================
raw_floor_data = {}
ingestion_audit = []

t_start = time.time()

for floor_num in range(1, 8):
    floor_key = f"Floor{floor_num}"
    f18_path = os.path.join(raw_source_dir, f"2018Floor{floor_num}.csv")
    if not os.path.exists(f18_path): f18_path = os.path.join(data_dir, f"2018Floor{floor_num}.csv")
    f19_path = os.path.join(raw_source_dir, f"2019Floor{floor_num}.csv")
    if not os.path.exists(f19_path): f19_path = os.path.join(data_dir, f"2019Floor{floor_num}.csv")
    
    frames = []
    total_raw_rows = 0
    
    for p in [f18_path, f19_path]:
        if os.path.exists(p):
            # Read header to isolate target columns
            header_df = pd.read_csv(p, nrows=2)
            all_cols = header_df.columns.tolist()
            
            q1_cols = ['Date'] + [
                c for c in all_cols 
                if ('_Light(kW)' in c) or ('_Plug(kW)' in c)
            ]
            
            df_raw = pd.read_csv(p, usecols=q1_cols, low_memory=False)
            total_raw_rows += len(df_raw)
            frames.append(df_raw)
        else:
            print(f"[WARN] File not found: {p}")
            
    if frames:
        combined_df = pd.concat(frames, axis=0, ignore_index=True)
        raw_floor_data[floor_key] = combined_df
        
        light_cols = [c for c in combined_df.columns if '_Light(kW)' in c]
        plug_cols = [c for c in combined_df.columns if '_Plug(kW)' in c]
        mem_mb = combined_df.memory_usage(deep=True).sum() / (1024 * 1024)
        
        ingestion_audit.append({
            'Floor': floor_key,
            'Raw_Records': total_raw_rows,
            'Lighting_Circuits': len(light_cols),
            'Plug_Circuits': len(plug_cols),
            'Memory_Footprint_MB': round(mem_mb, 2)
        })

audit_df = pd.DataFrame(ingestion_audit)
elapsed = time.time() - t_start

print(f"[SUCCESS] Ingested raw telemetry for all 7 floors in {elapsed:.2f} seconds.")
display(audit_df)


[SUCCESS] Ingested raw telemetry for all 7 floors in 15.27 seconds.


,Floor,Raw_Records,Lighting_Circuits,Plug_Circuits,Memory_Footprint_MB
0,Floor1,790560,4,3,99.52
1,Floor2,790560,4,4,105.55
2,Floor3,790560,5,5,117.61
3,Floor4,790560,5,5,117.61
4,Floor5,790560,5,5,117.61
5,Floor6,689129,5,5,102.52
6,Floor7,790560,5,5,117.61


## 3. Persisting Intermediate Target Telemetry
We save the isolated target circuits for each floor into `../data/raw_Floor{1-7}.csv`.

In [3]:
# ==============================================================================
# 3. SAVE RAW INTERMEDIATE TABLES FOR NOTEBOOK 1C
# ==============================================================================
for floor_key, df in raw_floor_data.items():
    out_path = os.path.join(data_dir, f"raw_{floor_key}.csv")
    df.to_csv(out_path, index=False)
    print(f"[SAVED] {floor_key} -> {out_path} ({len(df):,} records)")

print("\n[SUMMARY] Ingestion complete. Proceed to 'qn1c_data_cleaning.ipynb'.")

[SAVED] Floor1 -> c:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data\raw_Floor1.csv (790,560 records)


[SAVED] Floor2 -> c:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data\raw_Floor2.csv (790,560 records)


[SAVED] Floor3 -> c:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data\raw_Floor3.csv (790,560 records)


[SAVED] Floor4 -> c:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data\raw_Floor4.csv (790,560 records)


[SAVED] Floor5 -> c:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data\raw_Floor5.csv (790,560 records)


[SAVED] Floor6 -> c:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data\raw_Floor6.csv (689,129 records)


[SAVED] Floor7 -> c:\Users\65964\OneDrive\Desktop\IS4001\proj\qn1\data\raw_Floor7.csv (790,560 records)

[SUMMARY] Ingestion complete. Proceed to 'qn1c_data_cleaning.ipynb'.
